In [4]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [6]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [10]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [11]:
# Generate the dataset and inspect it
dataset = generate_dataset()
dataset

[{'task': "Write a Python function that extracts the AWS account ID from an ARN string (e.g., 'arn:aws:s3:::my-bucket')"},
 {'task': "Create a JSON object representing an AWS IAM policy that allows read-only access to a specific S3 bucket named 'my-data-bucket'"},
 {'task': 'Write a regular expression that matches valid AWS S3 bucket names (3-63 characters, lowercase letters, numbers, and hyphens only, cannot start or end with a hyphen)'}]

In [13]:
import json

# Save the dataset to a file for later use during evaluation
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

## Проведение оценки

Конвейер оценки состоит из трёх функций:

- `run_prompt` — объединяет тестовый случай с шаблоном промта и отправляет запрос Клоду.
- `run_test_case` — запускает один тестовый случай и оценивает результат.
- `run_eval` — координирует обработку всего набора данных.

In [18]:
# Объединяет шаблон промта с входными данными тестового случая
# и возвращает ответ модели
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [17]:
# Запускает один тестовый случай и возвращает результат с оценкой
def run_test_case(test_case):
    output = run_prompt(test_case)

    # TODO: реализовать логику оценки
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
    }

In [15]:
# Обрабатывает все тестовые случаи из набора данных
def run_eval(dataset):
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [19]:
# Загружаем набор данных из файла и запускаем оценку.
# Полная обработка набора может занять около 30 секунд.
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [20]:
# Выводим результаты: ответ модели, исходный тестовый случай и оценку
print(json.dumps(results, indent=2, ensure_ascii=False))

[
  {
    "output": "# AWS Account ID Extraction from ARN\n\nHere's a Python function that extracts the AWS account ID from an ARN string:\n\n```python\ndef extract_account_id_from_arn(arn: str) -> str:\n    \"\"\"\n    Extracts the AWS account ID from an ARN string.\n    \n    ARN format: arn:partition:service:region:account-id:resource\n    \n    Args:\n        arn: The ARN string to parse\n        \n    Returns:\n        The AWS account ID (empty string if not present)\n        \n    Raises:\n        ValueError: If the ARN format is invalid\n        \n    Examples:\n        >>> extract_account_id_from_arn('arn:aws:iam::123456789012:user/Development/product_1234/*')\n        '123456789012'\n        \n        >>> extract_account_id_from_arn('arn:aws:s3:::my-bucket')\n        ''\n        \n        >>> extract_account_id_from_arn('arn:aws:ec2:us-east-1:123456789012:instance/i-1234567890abcdef0')\n        '123456789012'\n    \"\"\"\n    if not isinstance(arn, str):\n        raise ValueEr